# Real Data ST Temporal-Geometry Ablation: Hybrid vs Column V3

This notebook keeps the richer `hour_spatial` GLS mean design fixed and tests whether the remaining Hybrid-vs-Column discrepancy is driven by temporal conditioning geometry.

The key ablations are:

- Hybrid baseline: same-time max-min NN plus lag self/local/fresh shifted neighbors.
- Hybrid local-only: remove fresh shifted lag neighbors.
- Hybrid self-only temporal: keep only lag self anchors beyond same-time spatial NN.
- Column baseline: reverse-L same-time and lag reverse-L neighbors, no lag self anchor.
- Column lag-self: same as Column baseline, but each lag first includes the same grid location as a temporal anchor.

Interpretation target: if Column moves toward Hybrid after adding lag self, or Hybrid moves toward Column after removing fresh shifted lag neighbors, the ST discrepancy is mainly temporal-conditioning geometry rather than spatial trend.


In [1]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_st_trend_050726 import HybridSTTrendVecchiaFit, ColumnSTTrendVecchiaFit

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
torch: 2.5.1
device: cpu


In [2]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: [2] = July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]
MEAN_DESIGNS = ['hour_spatial']

HYBRID_VARIANTS = [
    {
        'variant': 'hybrid_base',
        'model': 'HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exactloc',
        'note': 'baseline: same-time A20 + lag self/local/fresh shifted neighbors',
        'nheads': 0, 'limit_A': 20, 'lag1_local_count': 8, 'lag1_fresh_count': 4,
        'lag2_local_count': 4, 'lag2_fresh_count': 3, 'daily_stride': 2,
        'lag1_lon_offset': 0.063, 'spatial_coords_for_shift': None,
    },
    {
        'variant': 'hybrid_local_only',
        'model': 'HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p063_exactloc',
        'note': 'remove fresh shifted lag neighbors; keep lag self and local lag NN',
        'nheads': 0, 'limit_A': 20, 'lag1_local_count': 8, 'lag1_fresh_count': 0,
        'lag2_local_count': 4, 'lag2_fresh_count': 0, 'daily_stride': 2,
        'lag1_lon_offset': 0.063, 'spatial_coords_for_shift': None,
    },
    {
        'variant': 'hybrid_self_only_temporal',
        'model': 'HybridSTGeom_SelfOnly_A20_L00F00_C0F00_Op0p063_exactloc',
        'note': 'same-time A20 plus only lag self anchors; no lag local/fresh neighbors',
        'nheads': 0, 'limit_A': 20, 'lag1_local_count': 0, 'lag1_fresh_count': 0,
        'lag2_local_count': 0, 'lag2_fresh_count': 0, 'daily_stride': 2,
        'lag1_lon_offset': 0.063, 'spatial_coords_for_shift': None,
    },
]

COLUMN_VARIANTS = [
    {
        'variant': 'column_no_self',
        'model': 'ColumnSTGeom_Up2_Right3_Down14_Lag2_NoSelf_head0_realloc',
        'note': 'baseline column V3: per-lag reverse-L candidates, no lag self anchor',
        'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3,
        'per_lag_conditioning_count': 14, 'lag_count': 2, 'include_lag_self': False,
    },
    {
        'variant': 'column_with_self',
        'model': 'ColumnSTGeom_Up2_Right3_Down14_Lag2_WithSelf_head0_realloc',
        'note': 'column V3 plus lag self anchor; self replaces one reverse-L candidate within each lag cap',
        'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3,
        'per_lag_conditioning_count': 14, 'lag_count': 2, 'include_lag_self': True,
    },
]

for spec in COLUMN_VARIANTS:
    spec['target_chunk_size'] = 512 if DEVICE.type == 'cpu' else 4096

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_st_temporal_geometry_ablation_050726_results.csv'

print('days:', DAY_IDX_LIST)
print('mean designs:', MEAN_DESIGNS)
print('output:', OUT_CSV)
print('hybrid variants:', [s['variant'] for s in HYBRID_VARIANTS])
print('column variants:', [s['variant'] for s in COLUMN_VARIANTS])


days: [2]
mean designs: ['hour_spatial']
output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/real_st_temporal_geometry_ablation_050726_results.csv
hybrid variants: ['hybrid_base', 'hybrid_local_only', 'hybrid_self_only_temporal']
column variants: ['column_no_self', 'column_with_self']


In [3]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def hybrid_tail_count(model):
    total = 0
    for x in (getattr(model, 'X_A', None), getattr(model, 'X_AB', None), getattr(model, 'X_ABC', None)):
        if x is not None:
            total += int(x.shape[0])
    return total


def st_diag(model):
    batched = getattr(model, 'Batched_Groups', None)
    if batched:
        group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in batched], dtype=np.int64)
        m_sizes = np.asarray([int(g['max_m']) for g in batched], dtype=np.int64)
        return {
            'n_batches': int(len(batched)),
            'largest_batch_n': int(group_sizes.max()),
            'mean_m': float(m_sizes.mean()),
            'max_m': int(m_sizes.max()),
            'n_tails': int(group_sizes.sum()),
            'n_templates': np.nan,
        }
    return {
        'n_batches': np.nan,
        'largest_batch_n': np.nan,
        'mean_m': np.nan,
        'max_m': np.nan,
        'n_tails': hybrid_tail_count(model),
        'n_templates': np.nan,
    }


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def save_results(rows):
    df = pd.DataFrame(rows)
    round_df(df).to_csv(OUT_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    return df


def result_row(date_str, day_idx, smooth, mean_design, model_name, kernel, loss, fit_iter, pre_s, fit_s, n_valid, est, diag):
    row = {
        'date_str': date_str,
        'day_idx': int(day_idx),
        'smooth': float(smooth),
        'mean_design': mean_design,
        'model': model_name,
        'kernel': kernel,
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': round(pre_s, 3),
        'fit_s': round(fit_s, 3),
        'total_s': round(pre_s + fit_s, 3),
        'n_valid': int(n_valid),
    }
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    row.update(diag)
    return row

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])


Initial log params: [3.9558, 1.3863, 0.4463, -3.5835, 0.0218, -0.1689, -1.3984]


In [4]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)

del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('grid lat:', grid_coords_ordered[:, 0].min(), grid_coords_ordered[:, 0].max())
print('grid lon:', grid_coords_ordered[:, 1].min(), grid_coords_ordered[:, 1].max())


--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Month time slots loaded then trimmed: 248 -> 8
Grid cells: 18,126
grid lat: -2.9720000000000044 2.0
grid lon: 121.04600000000188 131.0


In [5]:
def load_day_map(day_idx, keep_ori=True):
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=keep_ori,
    )
    return {k: v.to(DEVICE) for k, v in day_map.items()}, day_keys


def hybrid_nominal_m(spec):
    return int(spec['limit_A'] + 1 + spec['lag1_local_count'] + spec['lag1_fresh_count'] + 1 + spec['lag2_local_count'] + spec['lag2_fresh_count'])


def column_nominal_m(spec):
    return int(spec['per_lag_conditioning_count'] * (spec['lag_count'] + 1))


def fit_hybrid_st(day_idx, smooth, mean_design, spec):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"HYBRID ST | {spec['variant']} | mean={mean_design} | DAY {date_str} | smooth={smooth} | valid={n_valid:,}")
    print('note:', spec['note'])

    params = make_params()
    model = HybridSTTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        nns_map=nns_map_full,
        mm_cond_number=MM_COND_NUMBER,
        nheads=spec['nheads'],
        limit_A=spec['limit_A'],
        limit_B_local=spec['lag1_local_count'],
        limit_C_local=spec['lag2_local_count'],
        daily_stride=spec['daily_stride'],
        spatial_coords=spec['spatial_coords_for_shift'],
        lag1_lon_offset=spec['lag1_lon_offset'],
        lag1_fresh_count=spec['lag1_fresh_count'],
        lag2_fresh_count=spec['lag2_fresh_count'],
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_params(out)

    row = result_row(date_str, day_idx, smooth, mean_design, spec['model'], 'hybrid_st_temporal_ablation', float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag)
    row.update({
        'variant': spec['variant'],
        'note': spec['note'],
        'total_conditioning_nominal': hybrid_nominal_m(spec),
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'exact_t0_source_coords',
        'limit_A': spec['limit_A'],
        'lag1_local_count': spec['lag1_local_count'],
        'lag1_fresh_count': spec['lag1_fresh_count'],
        'lag2_local_count': spec['lag2_local_count'],
        'lag2_fresh_count': spec['lag2_fresh_count'],
        'include_lag_self': True,
    })
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


def fit_column_st(day_idx, smooth, mean_design, spec):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"COLUMN ST REALLOC | {spec['variant']} | mean={mean_design} | DAY {date_str} | smooth={smooth} | valid={n_valid:,}")
    print('note:', spec['note'])

    params = make_params()
    model = ColumnSTTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=spec['head_right_cols'],
        above_count=spec['above_count'],
        right_col_count=spec['right_col_count'],
        per_lag_conditioning_count=spec['per_lag_conditioning_count'],
        lag_count=spec['lag_count'],
        include_lag_self=spec['include_lag_self'],
        target_chunk_size=spec['target_chunk_size'],
        use_data_coords_for_offsets=True,
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_params(out)

    row = result_row(date_str, day_idx, smooth, mean_design, spec['model'], 'column_st_temporal_ablation_realloc', float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag)
    row.update({
        'variant': spec['variant'],
        'note': spec['note'],
        'total_conditioning_nominal': column_nominal_m(spec),
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'regular_grid_reverse_l_scan',
        'head_right_cols': spec['head_right_cols'],
        'above_count': spec['above_count'],
        'right_col_count': spec['right_col_count'],
        'per_lag_conditioning_count': spec['per_lag_conditioning_count'],
        'lag_count': spec['lag_count'],
        'include_lag_self': spec['include_lag_self'],
    })
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


In [6]:
# Run temporal-geometry ablation.
rows = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        for mean_design in MEAN_DESIGNS:
            for spec in HYBRID_VARIANTS:
                rows.append(fit_hybrid_st(day_idx, smooth, mean_design, spec))
                save_results(rows)
            for spec in COLUMN_VARIANTS:
                rows.append(fit_column_st(day_idx, smooth, mean_design, spec))
                save_results(rows)

results = pd.DataFrame(rows)
print('Saved:', OUT_CSV)
display(round_df(results))



HYBRID ST | hybrid_base | mean=hour_spatial | DAY 20240703 | smooth=0.5 | valid=140,352
note: baseline: same-time A20 + lag self/local/fresh shifted neighbors
Pre-computing HybridVecchia (smooth=0.5) [A=20, AB=33, ABC=41, B=local8+fresh4, C=local4+fresh3, lag1_offset=0.0630, stored=1]... [Mean Lat: -0.5041] [SetC=True] Done. (Heads=0, Tails A/AB/ABC=18067/17670/104615)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/5 / Loss: 1.550974 ---
  Param 0: Value=4.4451, Grad=0.0014835465558599454
  Param 1: Value=1.8019, Grad=0.0006429944279744233
  Param 2: Value=0.2808, Grad=-0.0022321209482462
  Param 3: Value=-4.1255, Grad=-0.0011167462119029744
  Param 4: Value=-0.0554, Grad=0.0004623655946133058
  Param 5: Value=-0.2482, Grad=0.0013635885804542747
  Param 6: Value=-0.8323, Grad=-0.0030594863376626792
  Max Abs Grad: 3.059486e-03
------------------------------
--- Step 2/5 / Loss: 1.499489 ---
  Param 0: Value=3.6772, Grad=-0.0044670572689129145
  Param 1: Value=1.0551, Gr

,date_str,day_idx,smooth,mean_design,model,kernel,loss,fit_iter_raw,fit_steps_reported,precompute_s,...,lag1_local_count,lag1_fresh_count,lag2_local_count,lag2_fresh_count,include_lag_self,head_right_cols,above_count,right_col_count,per_lag_conditioning_count,lag_count
0,20240703,2,0.5,hour_spatial,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,hybrid_st_temporal_ablation,1.4912,2,3,8.762,...,8.0,4.0,4.0,3.0,True,NaN,NaN,NaN,NaN,NaN
1,20240703,2,0.5,hour_spatial,HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p06...,hybrid_st_temporal_ablation,1.5082,2,3,8.469,...,8.0,0.0,4.0,0.0,True,NaN,NaN,NaN,NaN,NaN
2,20240703,2,0.5,hour_spatial,HybridSTGeom_SelfOnly_A20_L00F00_C0F00_Op0p063...,hybrid_st_temporal_ablation,1.7336,4,5,8.113,...,0.0,0.0,0.0,0.0,True,NaN,NaN,NaN,NaN,NaN
3,20240703,2,0.5,hour_spatial,ColumnSTGeom_Up2_Right3_Down14_Lag2_NoSelf_hea...,column_st_temporal_ablation_realloc,1.3818,3,4,2.103,...,NaN,NaN,NaN,NaN,False,0.0,2.0,3.0,14.0,2.0
4,20240703,2,0.5,hour_spatial,ColumnSTGeom_Up2_Right3_Down14_Lag2_WithSelf_h...,column_st_temporal_ablation_realloc,1.3819,2,3,2.119,...,NaN,NaN,NaN,NaN,True,0.0,2.0,3.0,14.0,2.0


In [7]:
summary_cols = [
    'date_str', 'smooth', 'mean_design', 'variant', 'model', 'kernel', 'total_conditioning_nominal',
    'loss', 'precompute_s', 'fit_s', 'total_s', 'n_valid', 'n_batches', 'n_tails',
    'mean_m', 'max_m', 'largest_batch_n', 'include_lag_self',
    'lag1_local_count', 'lag1_fresh_count', 'lag2_local_count', 'lag2_fresh_count',
    'above_count', 'right_col_count', 'per_lag_conditioning_count',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time', 'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'coords_used_for_covariance', 'coords_used_for_shift_lookup', 'note',
]
existing = [c for c in summary_cols if c in results.columns]
display(round_df(results[existing].sort_values(['kernel', 'variant'])))

param_rows = []
for _, row in results.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'mean_design': row['mean_design'],
            'variant': row['variant'],
            'model': row['model'],
            'parameter': p,
            'estimate': row[f'est_{p}'],
        })
param_est = pd.DataFrame(param_rows)
display(round_df(param_est))


,date_str,smooth,mean_design,variant,model,kernel,total_conditioning_nominal,loss,precompute_s,fit_s,...,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget,coords_used_for_covariance,coords_used_for_shift_lookup,note
3,20240703,0.5,hour_spatial,column_no_self,ColumnSTGeom_Up2_Right3_Down14_Lag2_NoSelf_hea...,column_st_temporal_ablation_realloc,42,1.3818,2.103,267.691,...,12.4163,0.1437,0.1698,1.2904,-0.0683,-0.2343,1.2488,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan,baseline column V3: per-lag reverse-L candidat...
4,20240703,0.5,hour_spatial,column_with_self,ColumnSTGeom_Up2_Right3_Down14_Lag2_WithSelf_h...,column_st_temporal_ablation_realloc,42,1.3819,2.119,233.722,...,12.4472,0.1443,0.1707,1.2470,-0.0612,-0.2265,1.2523,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan,column V3 plus lag self anchor; self replaces ...
0,20240703,0.5,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,hybrid_st_temporal_ablation,41,1.4912,8.762,260.856,...,12.4887,0.2503,0.3032,1.8253,-0.0632,-0.2568,2.6088,Source_Latitude/Source_Longitude,exact_t0_source_coords,baseline: same-time A20 + lag self/local/fresh...
1,20240703,0.5,hour_spatial,hybrid_local_only,HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p06...,hybrid_st_temporal_ablation,34,1.5082,8.469,202.007,...,12.6299,0.2794,0.3556,1.8604,-0.0512,-0.2122,2.8041,Source_Latitude/Source_Longitude,exact_t0_source_coords,remove fresh shifted lag neighbors; keep lag s...
2,20240703,0.5,hour_spatial,hybrid_self_only_temporal,HybridSTGeom_SelfOnly_A20_L00F00_C0F00_Op0p063...,hybrid_st_temporal_ablation,22,1.7336,8.113,123.263,...,12.7226,0.4577,0.5956,2.3417,-0.0567,-0.2193,3.7622,Source_Latitude/Source_Longitude,exact_t0_source_coords,same-time A20 plus only lag self anchors; no l...


,date_str,mean_design,variant,model,parameter,estimate
0,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,sigmasq,12.4887
1,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,range_lat,0.2503
2,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,range_lon,0.3032
3,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,range_time,1.8253
4,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,advec_lat,-0.0632
5,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,advec_lon,-0.2568
6,20240703,hour_spatial,hybrid_base,HybridSTGeom_Base_A20_L08F04_C4F03_Op0p063_exa...,nugget,2.6088
7,20240703,hour_spatial,hybrid_local_only,HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p06...,sigmasq,12.6299
8,20240703,hour_spatial,hybrid_local_only,HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p06...,range_lat,0.2794
9,20240703,hour_spatial,hybrid_local_only,HybridSTGeom_LocalOnly_A20_L08F00_C4F00_Op0p06...,range_lon,0.3556
